# Setup and manual configuration
Mounting Google Drive, cloning the repository from GitHub, setting up input/output paths, and installing specific dependencies inline.
Modify the variables in this cell before starting a new experiment.

In [1]:
import os
from google.colab import drive

# Mount Google Drive for datasets and results
drive.mount('/content/drive')

# ==========================================
# --- MANUAL EXPERIMENT CONFIGURATION ---
# ==========================================
GIT_REPO_URL = 'https://github.com/EmanuelePietroCometti/SuperSimpleNet.git'
# IMPORTANT: set this to YOUR augmentation branch (the one that carries these
# augmentation changes) before running, otherwise Colab clones a branch without them.
BRANCH_NAME = 'augmentation'
REPO_PATH = '/content/SuperSimpleNet'
DATA_PATH = '/content/drive/MyDrive/Tesi/datasets/MVTec'
SAVE_PATH = '/content/drive/MyDrive/Tesi/experiments/results_SuperSimpleNet/run_custom_05_optimizedParameter'
CATEGORY = 'Nero_SEMISUPERVISED_dustValidationAndTrain'
SETUP_NAME = 'superSimpleNet_optimizedParameter'
MODE = 'sup'

# Augmentation ablation switch (path relative to the repo root, or None):
#   'configs/aug_off.json'    -> NO augmentation (clean baseline)
#   'configs/aug_full.json'   -> all families on (photometric + geometric)
#   'configs/aug_affine.json' -> single-family ablation (see configs/EXPERIMENTS.md)
#   None                       -> same as aug_off.json
AUG_CONFIG = 'configs/aug_off.json'
# ==========================================

if not os.path.exists(REPO_PATH):
    print(f">>> Cloning branch '{BRANCH_NAME}' from GitHub...")
    # The -b flag forces Git to clone and checkout the specified branch immediately
    !git clone -b {BRANCH_NAME} {GIT_REPO_URL} {REPO_PATH}
else:
    print(f">>> Repository already cloned. Updating branch '{BRANCH_NAME}'...")
    os.chdir(REPO_PATH)
    # Ensure the local repository is explicitly on the target branch and pull updates
    !git checkout {BRANCH_NAME}
    !git pull origin {BRANCH_NAME}

# Change to the working directory
os.chdir(REPO_PATH)
print(f"Working directory set to: {os.getcwd()}")
print(f"Augmentation config: {AUG_CONFIG or 'OFF (baseline)'}")

# Directly create the output folder on Drive
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"Output folder ready on Drive: {SAVE_PATH}")

# Install standard dependencies
!pip install tqdm anomalib==0.7

# Install specific PyTorch version with CUDA 11.8 support
!pip install torch==2.1.0+cu118 torchvision==0.16.0+cu118 --extra-index-url https://download.pytorch.org/whl/cu118

# Optional: Install wandb for experiment tracking
!pip install wandb optuna

# Install onnx export/conversion dependencies
# - onnx, onnxscript: required for the default fp32 export (export_onnx.py)
# - onnxruntime: required for --precision int8 (static/calibrated quantization)
# - onnxconverter-common: required for --precision fp16
!pip install onnx onnxscript onnxruntime onnxconverter-common

# CRITICAL: keep NumPy < 2.0 -- anomalib 0.7 pulls imgaug, which uses np.sctypes
# (removed in NumPy 2.0). Pin it LAST so no earlier install bumps it back to 2.x,
# otherwise `import anomalib` crashes with "np.sctypes was removed".
!pip install "numpy<2.0"

Mounted at /content/drive
>>> Cloning branch 'augmentation' from GitHub...
Cloning into '/content/SuperSimpleNet'...
remote: Enumerating objects: 464, done.
remote: Counting objects: 100% (142/142), done.
remote: Compressing objects: 100% (46/46), done.
remote: Total 464 (delta 118), reused 97 (delta 96), pack-reused 322 (from 1)
Receiving objects: 100% (464/464), 280.73 KiB | 6.53 MiB/s, done.
Resolving deltas: 100% (270/270), done.
Working directory set to: /content/SuperSimpleNet
Augmentation config: configs/aug_off.json
Output folder ready on Drive: /content/drive/MyDrive/Tesi/experiments/results_SuperSimpleNet/run_custom_05_optimizedParameter
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 349.7/349.7 kB 27.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.0/948.0 kB 66.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.7/529.7 kB 46.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.8/35.8 MB 74.

In [7]:
import os, shutil, time

# Source on Drive (whatever Setup pointed DATA_PATH at) -> local destination.
DRIVE_DATA = DATA_PATH
LOCAL_DATA = '/content/mvtec_local'
src = os.path.join(DRIVE_DATA, CATEGORY)
dst = os.path.join(LOCAL_DATA, CATEGORY)

if not os.path.isdir(src):
    raise FileNotFoundError(f"Source category not found on Drive: {src}")

if os.path.isdir(dst) and os.listdir(dst):
    print(f">>> Local copy already present, skipping copy: {dst}")
else:
    os.makedirs(LOCAL_DATA, exist_ok=True)
    print(f">>> Copying {src}\n           -> {dst} ...")
    t0 = time.time()
    shutil.copytree(src, dst, dirs_exist_ok=True)
    print(f">>> Copy done in {time.time() - t0:.1f}s")

# Point the pipeline at the local copy for all downstream cells.
DATA_PATH = LOCAL_DATA
print(">>> DATA_PATH now ->", DATA_PATH)

>>> Copying /content/drive/MyDrive/Tesi/datasets/MVTec/Nero_SEMISUPERVISED_dustValidationAndTrain
           -> /content/mvtec_local/Nero_SEMISUPERVISED_dustValidationAndTrain ...


KeyboardInterrupt: 

# Hyperparameter Optimization (Optional)
Launch the fine-tuning script. It dynamically reads the parameters configured in Cell 1.

In [ ]:
import os
os.chdir(REPO_PATH)

print(f">>> Starting Hyperparameter Optimization on dataset: {CATEGORY}...")

!python hyperparameter_finetuning.py \
    --dataset mvtec \
    --category {CATEGORY} \
    --data_path {DATA_PATH} \
    --datasets_folder {DATA_PATH} \
    --results_save_path {SAVE_PATH} \
    --setup_name {SETUP_NAME} \
    --epochs 100 \
    --batch 4

# Training, Evaluation and ONNX Export
This single cell orchestrates the entire pipeline:
1. Runs `train.py` with the specified parameters.
2. Automatically locates the generated `.pt` (or `.ckpt`) weights file on Google Drive.
3. Passes the weights file to `eval.py` and `export_onnx.py`.
4. Organizes the generated ONNX models into a dedicated subfolder.

In [6]:
import os
import glob
import shutil

os.chdir(REPO_PATH)

# --- Resume guard (SK-RD4AD-style): don't redo a run that already finished. ---
# If this SAVE_PATH already contains trained weights, training is skipped and we
# go straight to eval/export. Set FORCE_RETRAIN=True to retrain from scratch.
FORCE_RETRAIN = False

existing_weights = glob.glob(os.path.join(SAVE_PATH, "**", "weights.pt"), recursive=True) + \
                   glob.glob(os.path.join(SAVE_PATH, "**", "*.pt"), recursive=True)
SHOULD_TRAIN = FORCE_RETRAIN or not existing_weights

# Build the augmentation flag: omit it entirely when AUG_CONFIG is None,
# so train.py falls back to the clean baseline (augmentation OFF).
AUG_FLAG = f'--aug_config {AUG_CONFIG}' if AUG_CONFIG else ''
print("Aug flag:", AUG_FLAG or '(none -> baseline)')

if not SHOULD_TRAIN:
    print(f">>> Existing weights found, SKIPPING training (resume): {existing_weights[0]}")
    print(">>> Set FORCE_RETRAIN = True above to retrain from scratch.")
else:
    print(f">>> Starting Training on {CATEGORY}...")
    print(f">>> Saving results directly to: {SAVE_PATH}\n")

    # Lancia train.py iniettando le variabili Python
    !python train.py \
        --dataset mvtec \
        --category {CATEGORY} \
        --mode {MODE} \
        --data_path {DATA_PATH} \
        --datasets_folder {DATA_PATH} \
        --results_save_path {SAVE_PATH} \
        --setup_name {SETUP_NAME} \
        --num_workers 1 \
        --backbone wide_resnet50_2 \
        --layers layer1 layer2 layer3 \
        --image_size 256 256 \
        --epochs 300 \
        --batch 4 \
        --perlin_thr 0.34628 \
        --noise_std 0.10580 \
        --seg_lr 2.087e-5 \
        --dec_lr 7.798e-4 \
        --adapt_lr 0.0001 \
        --patch_size 5 \
        --gamma 0.52232 \
        --eval_step_size 25 \
        --seed 42 \
        {AUG_FLAG}

print(f"\n>>> Searching for the generated weights in {SAVE_PATH}...")
# Cerca ricorsivamente qualsiasi file dei pesi generato dal training
weight_files = glob.glob(os.path.join(SAVE_PATH, "**", "*.pt"), recursive=True) + \
               glob.glob(os.path.join(SAVE_PATH, "**", "*.pth"), recursive=True) + \
               glob.glob(os.path.join(SAVE_PATH, "**", "*.ckpt"), recursive=True)

if not weight_files:
    print("[ERROR] No weights file found. The training process might have failed.")
else:
    # Prende il primo file dei pesi trovato
    WEIGHTS_FILE = weight_files[0]
    print(f">>> Found weights file: {WEIGHTS_FILE}")

    print("\n" + "="*40)
    print("--- STARTING EVALUATION ---")
    print("="*40)
    !python eval.py "{WEIGHTS_FILE}" \
    --dataset mvtec \
    --category {CATEGORY} \
    --datasets_folder {DATA_PATH} \
    --results_save_path {SAVE_PATH} \
    --image_size 512 512 \
    --batch 4 \
    --num_workers 1 \
    --seed 42 \
    --patch_size 5 \
    --layers layer1 layer2 layer3

    print("\n" + "="*40)

    print(f"\n>>> Pipeline completed successfully! ONNX models saved in: {onnx_dir}")

Aug flag: --aug_config configs/aug_off.json
>>> Starting Training on Nero_SEMISUPERVISED_dustValidationAndTrain...
>>> Saving results directly to: /content/drive/MyDrive/Tesi/experiments/results_SuperSimpleNet/run_custom_05_optimizedParameter

Starting training pipeline in SUP mode on dataset: mvtec
Training on Nero_SEMISUPERVISED_dustValidationAndTrain
Global seed set to 42
Downloading: "https://download.pytorch.org/models/wide_resnet50_2-95faca4d.pth" to /root/.cache/torch/hub/checkpoints/wide_resnet50_2-95faca4d.pth
100% 132M/132M [00:00<00:00, 246MB/s]
Resolution set to: (256, 256)
Augmentation config: {'enabled': False, 'dynamic_crop': False, 'equalize_p': 0.0, 'hflip_p': 0.0, 'vflip_p': 0.0, 'affine_deg': 0.0, 'affine_translate': 0.0, 'affine_scale': (1.0, 1.0), 'brightness': 0.0, 'contrast': 0.0, 'saturation': 0.0, 'hue': 0.0, 'grayscale_p': 0.0, 'blur_p': 0.0, 'blur_kernel': 3, 'blur_sigma': (0.1, 1.0), 'speckle_std': 0.0, 'seed': 0, 'schema_version': '1.0'}

DATASET DEBUG INFO

# Optional: Re-export ONNX at FP16 / INT8
The pipeline above always exports the model in FP32 first (`weights.onnx`, self-contained, dynamic batch, H/W fixed to the training `image_size`).

Run this cell to additionally try FP16 or INT8 without retraining: it just reconverts the already-exported FP32 graph and saves a separate file (`weights_fp16.onnx` / `weights_int8.onnx`), so you can compare speed/accuracy across precisions.

INT8 needs a calibration pass. Point `CALIBRATION_IMAGES_DIR` at a folder of real sample images (any size - they are resized in-graph) for a usable model; leave it as `None` to calibrate on random noise, which only lets you test that the pipeline runs, not real accuracy.</cell id="39050987">


In [ ]:
import os

os.chdir(REPO_PATH)

# Folder of real sample images for INT8 calibration (recommended). Leave as
# None to fall back to random noise (pipeline smoke-test only, not accurate).
CALIBRATION_IMAGES_DIR = None  # e.g. os.path.join(DATA_PATH, CATEGORY, 'train', 'good')

print("\n" + "="*40)
print("--- RE-EXPORTING ONNX AT FP16 ---")
print("="*40)
!python export_onnx.py "{WEIGHTS_FILE}" --image_size 512 512 --precision fp16

print("\n" + "="*40)
print("--- RE-EXPORTING ONNX AT INT8 ---")
print("="*40)
if CALIBRATION_IMAGES_DIR:
    !python export_onnx.py "{WEIGHTS_FILE}" --image_size 512 512 --precision int8 --calibration_images_dir "{CALIBRATION_IMAGES_DIR}"
else:
    !python export_onnx.py "{WEIGHTS_FILE}" --image_size 512 512 --precision int8

# Move the newly generated files into the same onnx/ folder as the fp32 export
onnx_dir = os.path.join(SAVE_PATH, "onnx")
os.makedirs(onnx_dir, exist_ok=True)
import glob, shutil
for onnx_file in glob.glob(os.path.join(SAVE_PATH, "**", "*.onnx"), recursive=True):
    if os.path.dirname(onnx_file) != onnx_dir:
        shutil.move(onnx_file, os.path.join(onnx_dir, os.path.basename(onnx_file)))

print(f"\n>>> FP16/INT8 ONNX models saved in: {onnx_dir}")

In [ ]:
from google.colab import runtime
runtime.disconnect()